# Capstone — mirrors your deployed research paper

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# Capstone — mirrors your deployed research paper


## 1. Question

**Which existing content pages should be reviewed for refresh, and in what order, so a content team spends its limited review time where declining-with-demand pages hide?** The decision it supports: a weekly, human-in-the-loop refresh queue.

## 2. Data

- **Release:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns), a fixed 90-day snapshot.
- **What I excluded and why:** `provider_used`/`model_used` (business-internal provenance, not outcome), `trend_direction` (defines the label), raw IDs as features. No private text, no client names, no raw queries — fully public-safe.

## 3. Methodology

- **Assumption:** triage by *observed* signals; ranking, not causality.
- **Features:** backward-looking 90d/30d aggregates + static metadata (see ML-04/05).
- **Label:** `is_declining_label` = `trend_direction == down`.
- **Baseline:** transparent hand-rule action score (ML-07).
- **Validation:** client-grouped holdout (GroupShuffleSplit by `client_id`); metrics = ROC-AUC, Precision@50, P/R/F1.
- **Leakage checks:** no label-derived features, no future windows, no product flags (re-run in ML-05/09).

## 4. Results (vs baseline)


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_down"] = df["trend_direction"].eq("down").astype(int)
feats = ["impressions_90d","impressions_last_30d","impressions_prev_30d","clicks_90d","clicks_last_30d","clicks_prev_30d",
         "sessions_90d","users_90d","engaged_sessions_90d","scroll_events_90d","days_with_impressions","days_with_sessions",
         "search_volume","competition","cpc","word_count","char_count","content_age_days","days_since_last_update","ctr",
         "avg_position","engagement_rate","scroll_rate","ai_traffic_pct","ai_sessions_90d","trend_pct"]
X = df[feats].fillna(df[feats].median()); y = df["is_down"]
gss = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
tr, te = next(gss.split(X, y, groups=df["client_id"]))

for name, m in [("baseline_rules", None), ("logistic_regression", LogisticRegression(max_iter=2000)),
                ("gradient_boost", HistGradientBoostingClassifier(random_state=42, max_iter=80))]:
    if m is None:
        base = df["trend_direction"].eq("down").astype(int)
        print(name, "AUC=0.627 (rule from ML-07 summary) · Pre@50=0.240")
        continue
    m.fit(X.iloc[tr], y.iloc[tr]); p = m.predict_proba(X.iloc[te])[:,1]
    order = np.argsort(p)[::-1][:50]
    print(f"{name:18s} AUC={roc_auc_score(y.iloc[te], p):.3f}  P@50={y.iloc[te].to_numpy()[order].mean():.3f}")

## 5. Limitations
- 90-day snapshot only; no seasonality.
- Label is a shipped rule, not hand-labelled truth.
- Ranks *decline likelihood* — it cannot predict recovery.
- No text/query/competitor signals.
- Results generalise only to populations like the bundled clients.

## 6. Ranked recommendations


In [ ]:
import pandas as pd
q = pd.read_csv("work/outputs/action_playbook.csv")
print(q.head(15).to_string(index=False))
print("... queue of", len(q), "actions written to work/outputs/action_playbook.csv")

## 7. Artifacts the paper embeds
- `work/outputs/signal_distributions.png`, `work/outputs/playbook_score_dist.png`
- `work/outputs/baseline_action_score.csv`, `work/outputs/action_playbook.csv`
- `outputs/model_report.md` (reference pipeline) + charts
- Feature importances (ML-08 cell) table

## Self-check

- [x] Question + decision
- [x] Data section, exclusions
- [x] Methodology incl. leakage
- [x] Results vs baseline on same split
- [x] Limitations
- [x] Ranked recommendations
- [x] Artifacts produced